# Chapter 10: Creating Text Embedding Models - Hard Tasks

This notebook covers advanced multi-stage training pipelines: Augmented SBERT, TSDAE implementation, domain adaptation, and combined training strategies.

**Note on Code Organization**: Like in previous hard task notebooks, we use functions extensively because:
- **Reusability**: Call the same logic multiple times without copying code
- **Modularity**: Each function handles one specific task (train cross-encoder, label data, etc.)
- **Real-world practice**: Production systems always use functions for maintainability
- **Testing**: Easy to test individual components independently

## Setup

Run all cells in this section to set up the environment and load the model.

Before running these cells, review the concepts from the main Chapter 10 notebook (00_Start_Here.ipynb).

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [1]:
%%capture
!pip install -q accelerate>=0.27.2 transformers>=4.38.2
!pip install -q sentence-transformers>=3.0.0 datasets>=2.18.0
!pip install -q nltk

### Model Loading

In [2]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, losses, InputExample, models
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.datasets import NoDuplicatesDataLoader

### Helper Functions

In [3]:
def create_evaluator():
    """Create standard evaluator for consistent evaluation"""
    val_sts = load_dataset('glue', 'stsb', split='validation')
    return EmbeddingSimilarityEvaluator(
        sentences1=val_sts["sentence1"],
        sentences2=val_sts["sentence2"],
        scores=[score/5 for score in val_sts["label"]],
        main_similarity="cosine"
    )

## Challenges

Complete the following tasks by implementing the starter code.

### Level: Hard

**About This Task:**

Augmented SBERT uses a cross-encoder to create silver labels for additional training data. We break this into 5 functions, each handling one step of the pipeline.

#### Hard Task 1: Augmented SBERT Pipeline

### Instructions

1. Study the 5-stage pipeline (prepare gold data → train cross-encoder → create silver pairs → label with cross-encoder → train bi-encoder)
2. Run baseline bi-encoder on gold data only
3. Complete the silver data labeling function
4. Train bi-encoder on gold + silver
5. Compare results to see if augmentation helped

**Stage 1: Prepare Gold Data**

Gold data is our high-quality labeled dataset.

In [4]:
def prepare_gold_data(num_samples=5_000):
    """
    Prepare gold dataset with ground-truth labels.

    Using a function here:
    - Can easily adjust data size
    - Consistent data preparation across experiments
    - Easy to swap datasets

    Args:
        num_samples: Number of examples to use

    Returns:
        gold_examples: List of InputExample for cross-encoder
        gold_df: Pandas DataFrame for easier data handling
    """
    print("Stage 1: Preparing gold data...")

    # Load MNLI data
    dataset = load_dataset("glue", "mnli", split="train").select(range(num_samples))

    # Convert to binary: entailment=1, neutral/contradiction=0
    mapping = {0: 1, 1: 0, 2: 0}

    # Create InputExample format for cross-encoder
    gold_examples = [
        InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
        for row in tqdm(dataset, desc="Creating gold examples")
    ]

    # Also create DataFrame for easier manipulation
    gold_df = pd.DataFrame({
        'sentence1': dataset['premise'],
        'sentence2': dataset['hypothesis'],
        'label': [mapping[label] for label in dataset['label']]
    })

    print(f"Gold dataset: {len(gold_examples)} examples")
    return gold_examples, gold_df

In [5]:
# Create gold data
gold_examples, gold_df = prepare_gold_data(5_000)

Stage 1: Preparing gold data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

Creating gold examples: 100%|██████████| 5000/5000 [00:00<00:00, 5106.46it/s]


Gold dataset: 5000 examples


**Stage 2: Train Cross-Encoder**

The cross-encoder will label our silver data.

In [6]:
def train_cross_encoder(gold_examples):
    """
    Train a cross-encoder on gold data.

    Separating this into a function:
    - Can reuse trained cross-encoder
    - Easy to swap architectures
    - Clear separation of concerns

    Args:
        gold_examples: List of InputExample objects

    Returns:
        Trained CrossEncoder model
    """
    print("\nStage 2: Training cross-encoder...")

    # Create data loader
    gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

    # Initialize cross-encoder
    cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)

    # Train
    cross_encoder.fit(
        train_dataloader=gold_dataloader,
        epochs=1,
        show_progress_bar=True,
        warmup_steps=50,
        use_amp=False
    )

    print("Cross-encoder training complete")
    return cross_encoder

In [7]:
# Train cross-encoder
cross_encoder = train_cross_encoder(gold_examples)


Stage 2: Training cross-encoder...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ads525 (ads525-ads525) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


Cross-encoder training complete


**Stage 3: Create Silver Data Pairs**

Generate new sentence pairs that need labels.

In [8]:
def create_silver_pairs(start_idx=5_000, end_idx=15_000):
    """
    Create unlabeled silver dataset from a different data range.

    Using a function allows:
    - Easy control of data range
    - Consistent format with gold data
    - Reusability for different splits

    Args:
        start_idx: Starting index in dataset
        end_idx: Ending index

    Returns:
        pairs: List of (sentence1, sentence2) tuples
        silver_dataset: Dataset object for later use
    """
    print("\nStage 3: Creating silver data pairs...")

    silver = load_dataset("glue", "mnli", split="train").select(range(start_idx, end_idx))

    # Create pairs (no labels yet)
    pairs = list(zip(silver['premise'], silver['hypothesis']))

    print(f"Created {len(pairs)} silver pairs")
    return pairs, silver

In [9]:
# Create silver pairs
silver_pairs, silver_raw = create_silver_pairs(5_000, 15_000)


Stage 3: Creating silver data pairs...
Created 10000 silver pairs


**Stage 4: Label Silver Data with Cross-Encoder**

Your task: Complete this function to use the cross-encoder for labeling.

In [10]:
def label_silver_data(cross_encoder, pairs, silver_dataset):
    """
    Use trained cross-encoder to label silver data.

    This function demonstrates the power of modular design:
    - Takes trained cross-encoder from Stage 2
    - Takes pairs from Stage 3
    - Produces labeled data for Stage 5
    - Each stage is independent and testable

    Args:
        cross_encoder: Trained CrossEncoder
        pairs: List of sentence pairs
        silver_dataset: Original dataset for reference

    Returns:
        silver_df: DataFrame with predicted labels
    """
    print("\nStage 4: Labeling silver data with cross-encoder...")

    # Fill in: Use cross_encoder.predict() with apply_softmax=True
    output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)

    # Take argmax to get predicted class
    predicted_labels = np.argmax(output, axis=1)

    # Create DataFrame
    silver_df = pd.DataFrame({
        "sentence1": silver_dataset["premise"],
        "sentence2": silver_dataset["hypothesis"],
        "label": predicted_labels
    })

    print(f"Labeled {len(silver_df)} silver examples")
    print(f"Label distribution: {silver_df['label'].value_counts().to_dict()}")

    return silver_df

In [11]:
# Label silver data
silver_df = label_silver_data(cross_encoder, silver_pairs, silver_raw)


Stage 4: Labeling silver data with cross-encoder...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Labeled 10000 silver examples
Label distribution: {0: 6586, 1: 3414}


**Stage 5: Train Bi-Encoder on Combined Data**

Now we train a bi-encoder (SBERT) on both gold and silver data.

In [12]:
def train_biencoder(combined_data, output_dir):
    """
    Train bi-encoder on gold + silver data.

    Separating this allows:
    - Training with different data combinations
    - A/B testing gold-only vs gold+silver
    - Reusing training logic

    Args:
        combined_data: DataFrame with all training data
        output_dir: Where to save model

    Returns:
        model: Trained SentenceTransformer
        results: Evaluation results
    """
    print(f"\nStage 5: Training bi-encoder on {len(combined_data)} examples...")

    # Remove duplicates
    combined_data = combined_data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")

    # Convert to Dataset
    train_dataset = Dataset.from_pandas(combined_data, preserve_index=False)

    # Create model
    embedding_model = SentenceTransformer('bert-base-uncased')

    # Loss function
    train_loss = losses.CosineSimilarityLoss(model=embedding_model)

    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=32,
        warmup_steps=100,
        fp16=True,
        logging_steps=100,
    )

    # Evaluator
    evaluator = create_evaluator()

    # Train
    trainer = SentenceTransformerTrainer(
        model=embedding_model,
        args=args,
        train_dataset=train_dataset,
        loss=train_loss,
        evaluator=evaluator
    )

    trainer.train()

    # Evaluate
    results = evaluator(embedding_model)

    return embedding_model, results

Train with gold + silver data.

In [13]:
# Combine gold and silver
combined = pd.concat([gold_df, silver_df], ignore_index=True, axis=0)

print(f"Combined dataset: {len(combined)} examples")
print(f"  Gold: {len(gold_df)}")
print(f"  Silver: {len(silver_df)}")

Combined dataset: 15000 examples
  Gold: 5000
  Silver: 10000


In [14]:
# Train augmented model
augmented_model, augmented_results = train_biencoder(combined, "augmented_sbert")

print("\nAugmented SBERT Results:")
print(f"Spearman Cosine: {augmented_results['spearman_cosine']:.4f}")


Stage 5: Training bi-encoder on 15000 examples...


Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Step,Training Loss
100,0.209800
200,0.143500
300,0.138000
400,0.137600



Augmented SBERT Results:
Spearman Cosine: 0.5993


### Task 1a: Train Baseline (Gold Only)

Compare against a model trained only on gold data.

In [15]:
# Your task: Train bi-encoder on gold data only
baseline_model, baseline_results = train_biencoder(gold_df, "gold_only_sbert")

print("\nGold-Only Results:")
print(f"Spearman Cosine: {baseline_results['spearman_cosine']:.4f}")


Stage 5: Training bi-encoder on 5000 examples...


Step,Training Loss
100,0.230500



Gold-Only Results:
Spearman Cosine: 0.5723


Compare results.

In [16]:
print("\n" + "="*60)
print("Augmented SBERT Comparison:")
print(f"Gold only:        {baseline_results['spearman_cosine']:.4f}")
print(f"Gold + Silver:    {augmented_results['spearman_cosine']:.4f}")
print(f"Improvement:      {augmented_results['spearman_cosine'] - baseline_results['spearman_cosine']:.4f}")


Augmented SBERT Comparison:
Gold only:        0.5723
Gold + Silver:    0.5993
Improvement:      0.0270


### Questions

1. Did silver data improve performance? Why would imperfect labels still help?

2. Which stage took the longest? How would you optimize this pipeline for production?

3. Why use 5 separate functions instead of one big script? Give 3 specific advantages.

**About This Task:**

TSDAE trains embeddings without labels by learning to reconstruct sentences from noisy versions. We build this using modular functions for data preparation, model creation, and training.

#### Hard Task 2: TSDAE Implementation

### Instructions

1. Study the 4-function pipeline (prepare sentences → add noise → create model → train with DAE loss)
2. Run TSDAE with default noise (deletion)
3. Implement custom noise function (word shuffling)
4. Compare TSDAE vs supervised training
5. Analyze when unsupervised learning helps

In [17]:
# Download NLTK tokenizer
import nltk
nltk.download('punkt', quiet=True)

True

**Function 1: Prepare Sentences**

Extract unique sentences for unsupervised training.

In [20]:
def prepare_sentences(num_samples=10_000):
    """
    Extract unique sentences for TSDAE training.

    Using a function:
    - Easy to control data size
    - Can swap data sources
    - Ensures deduplication

    Args:
        num_samples: Number of examples to load

    Returns:
        List of unique sentences
    """
    print("Preparing sentences for TSDAE...")

    # Load MNLI
    mnli = load_dataset("glue", "mnli", split="train").select(range(num_samples))

    # Combine premise and hypothesis into one flat list
    flat_sentences = list(mnli["premise"]) + list(mnli["hypothesis"])

    # Remove duplicates
    unique_sentences = list(set(flat_sentences))

    print(f"Collected {len(unique_sentences)} unique sentences")
    return unique_sentences

In [21]:
sentences = prepare_sentences(10_000)

Preparing sentences for TSDAE...
Collected 19713 unique sentences


**Function 2: Add Noise to Sentences**

Create damaged versions for the denoising task.

In [24]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [25]:
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

def create_noisy_data(sentences, noise_fn=None):
    """
    Create noisy versions of sentences.

    Separating noise creation:
    - Can test different noise strategies
    - Easy to visualize noise effects
    - Allows custom noise functions

    Args:
        sentences: List of clean sentences
        noise_fn: Optional custom noise function

    Returns:
        damaged_data: DenoisingAutoEncoderDataset
        dataset: Dataset for training
    """
    print("\nAdding noise to sentences...")

    if noise_fn:
        damaged_data = DenoisingAutoEncoderDataset(sentences, noise_fn=noise_fn)
    else:
        # Default: delete words with 60% probability
        damaged_data = DenoisingAutoEncoderDataset(sentences)

    # Convert to Dataset format
    train_dataset = {"damaged_sentence": [], "original_sentence": []}
    for data in tqdm(damaged_data, desc="Processing noisy data"):
        train_dataset["damaged_sentence"].append(data.texts[0])
        train_dataset["original_sentence"].append(data.texts[1])

    dataset = Dataset.from_dict(train_dataset)

    print(f"Created {len(dataset)} noisy examples")
    return damaged_data, dataset

In [26]:
# Create noisy data with default deletion
damaged_data, noisy_dataset = create_noisy_data(sentences)


Adding noise to sentences...


Processing noisy data: 100%|██████████| 19713/19713 [00:06<00:00, 3266.78it/s]

Created 19713 noisy examples


View examples of noise.

In [27]:
print("Noise examples:")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"  Original: {noisy_dataset[i]['original_sentence']}")
    print(f"  Damaged:  {noisy_dataset[i]['damaged_sentence']}")

Noise examples:

Example 1:
  Original: An article says the United States must not sacrifice its democratic ideals in order to sell a few more Big Macs.
  Damaged:  article says not sacrifice its ideals in Macs

Example 2:
  Original: 129 She began slowly, walking up and down the room, her head a little bent, and that slim, supple figure of hers swaying gently as she walked. 
  Damaged:  slowly down, her head bent, figure as she walked.

Example 3:
  Original: Thank you for playing.
  Damaged:  you


**Function 3: Create TSDAE Model**

Build model with encoder-decoder architecture.

In [28]:
def create_tsdae_model():
    """
    Create embedding model with CLS pooling for TSDAE.

    Using a function:
    - Consistent model architecture
    - Easy to swap base models
    - Clear separation from training

    Returns:
        SentenceTransformer model with CLS pooling
    """
    print("\nCreating TSDAE model...")

    # Create model components
    word_embedding_model = models.Transformer('bert-base-uncased')
    pooling_model = models.Pooling(
        word_embedding_model.get_word_embedding_dimension(),
        'cls'  # TSDAE uses CLS token
    )

    # Combine into SentenceTransformer
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

    print("Model created with CLS pooling")
    return model

In [29]:
tsdae_model = create_tsdae_model()


Creating TSDAE model...
Model created with CLS pooling


**Function 4: Train with TSDAE Loss**

Train model to reconstruct sentences from noisy input.

In [30]:
def train_tsdae(model, dataset, output_dir):
    """
    Train model with denoising autoencoder loss.

    This function shows the TSDAE training pattern:
    - Takes noisy dataset from Function 2
    - Uses DenoisingAutoEncoderLoss
    - Evaluates on downstream task

    Args:
        model: SentenceTransformer with CLS pooling
        dataset: Dataset with damaged and original sentences
        output_dir: Where to save model

    Returns:
        model: Trained model
        results: Evaluation results
    """
    print("\nTraining with TSDAE loss...")

    # Create DAE loss
    train_loss = losses.DenoisingAutoEncoderLoss(
        model,
        tie_encoder_decoder=True
    )

    # Move decoder to GPU
    train_loss.decoder = train_loss.decoder.to("cuda")

    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=100,
        fp16=True,
        logging_steps=100,
    )

    # Create evaluator
    evaluator = create_evaluator()

    # Train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        loss=train_loss,
        evaluator=evaluator
    )

    trainer.train()

    # Evaluate
    results = evaluator(model)

    return model, results

In [31]:
# Train TSDAE model
tsdae_model, tsdae_results = train_tsdae(tsdae_model, noisy_dataset, "tsdae_model")

print("\nTSDAE Results:")
print(f"Spearman Cosine: {tsdae_results['spearman_cosine']:.4f}")


Training with TSDAE loss...


Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Step,Training Loss
100,7.123600
200,4.933300
300,4.599300
400,4.479300
500,4.386900
600,4.282900
700,4.202500
800,4.139900
900,4.094900
1000,4.062100



TSDAE Results:
Spearman Cosine: 0.7461


### Task 2a: Custom Noise Function

Your task: Implement word shuffling noise.

In [32]:
import random

def shuffle_words(text):
    """
    Custom noise: shuffle word order.

    Args:
        text: Original sentence

    Returns:
        Shuffled sentence
    """
    # Fill in: Split text into words, shuffle, rejoin
    words = text.split()
    random.shuffle(words)
    return ' '.join(words)

# Test the noise function
test_sentence = "The quick brown fox jumps over the lazy dog"
print(f"Original: {test_sentence}")
print(f"Shuffled: {shuffle_words(test_sentence)}")

Original: The quick brown fox jumps over the lazy dog
Shuffled: lazy the over quick fox jumps The dog brown


In [33]:
# Create dataset with custom noise
_, shuffled_dataset = create_noisy_data(sentences[:5000], noise_fn=shuffle_words)

print("\nShuffle noise examples:")
for i in range(2):
    print(f"\nExample {i+1}:")
    print(f"  Original: {shuffled_dataset[i]['original_sentence']}")
    print(f"  Shuffled: {shuffled_dataset[i]['damaged_sentence']}")


Adding noise to sentences...


Processing noisy data: 100%|██████████| 5000/5000 [00:00<00:00, 164892.48it/s]

Created 5000 noisy examples

Shuffle noise examples:

Example 1:
  Original: An article says the United States must not sacrifice its democratic ideals in order to sell a few more Big Macs.
  Shuffled: sell order must States ideals its the article says a democratic United Macs. in few An more Big sacrifice to not

Example 2:
  Original: 129 She began slowly, walking up and down the room, her head a little bent, and that slim, supple figure of hers swaying gently as she walked. 
  Shuffled: as She supple began figure up of she slim, little walked. her 129 walking that gently head hers down slowly, and a and room, bent, the swaying


### Questions

1. How does TSDAE learn good embeddings without labels?

2. Compare deletion vs shuffling noise. Which is harder for the model to denoise?

3. When would you use TSDAE instead of supervised training?

**About This Task:**

Domain adaptation fine-tunes a general embedding model on domain-specific data to improve performance on that domain. We use a multi-stage pipeline.

#### Hard Task 3: Domain Adaptation Pipeline

### Instructions

1. Evaluate a general-purpose model on a domain-specific task
2. Collect domain-specific unlabeled data
3. Fine-tune with domain data using TSDAE
4. Re-evaluate on the domain task
5. Measure improvement from domain adaptation

**Stage 1: Baseline Evaluation**

Test a general model on a specialized domain.

In [39]:
def evaluate_on_domain(model, task_name="Banking77Classification"):
    """
    Evaluate model on domain-specific task.

    Using a function:
    - Consistent evaluation across experiments
    - Easy to swap tasks
    - Track performance over time

     Args:
        model: SentenceTransformer to evaluate
        task_name: MTEB task name

      Returns:
          accuracy: Main score from evaluation
      """
    from mteb import evaluate, get_task

    print(f"\nEvaluating on {task_name}...")

    # Get task object
    task = get_task(task_name)

    # Evaluate
    results = evaluate(model, tasks=[task])

    # Access with dot notation, not dictionary
    accuracy = results[0].scores["test"][0]["main_score"]
    print(f"Accuracy: {accuracy:.4f}")

    return accuracy


In [37]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 23.9 MB/s eta 0:00:00


In [40]:
# Load general-purpose model
general_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Evaluate baseline
baseline_accuracy = evaluate_on_domain(general_model)


Evaluating on Banking77Classification...


Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

Accuracy: 0.8004


**Stage 2: Collect Domain Data**

Your task: Prepare domain-specific sentences (we'll use banking domain).

In [41]:
def collect_domain_data(domain="banking", num_samples=5_000):
    """
    Collect domain-specific unlabeled sentences.

    In production:
    - Scrape domain documents
    - Use domain-specific corpora
    - Extract from task data (without labels)

    Args:
        domain: Domain name
        num_samples: Number of sentences

    Returns:
        List of domain sentences
    """
    print(f"\nCollecting {domain} domain data...")

    # For this example, use Banking77 training text (unlabeled)
    banking_data = load_dataset("banking77", split="train").select(range(num_samples))

    # Extract just the text (ignore labels)
    domain_sentences = list(set(banking_data["text"]))

    print(f"Collected {len(domain_sentences)} unique domain sentences")
    return domain_sentences

In [42]:
# Collect banking domain data
domain_sentences = collect_domain_data("banking", 5_000)


Collected 5000 unique domain sentences


**Stage 3: Domain Adaptation with TSDAE**

Fine-tune on domain data using unsupervised TSDAE.

In [43]:
def domain_adapt(model, domain_sentences, output_dir):
    """
    Adapt model to domain using TSDAE.

    This combines previous functions:
    - Uses noisy data creation from Task 2
    - Uses TSDAE training from Task 2
    - Applied to domain-specific data

    Args:
        model: Pre-trained SentenceTransformer
        domain_sentences: Domain-specific sentences
        output_dir: Where to save adapted model

    Returns:
        Adapted model
    """
    print("\nAdapting to domain...")

    # Create noisy data
    _, noisy_domain = create_noisy_data(domain_sentences)

    # Create DAE loss
    train_loss = losses.DenoisingAutoEncoderLoss(
        model,
        tie_encoder_decoder=True
    )
    train_loss.decoder = train_loss.decoder.to("cuda")

    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=50,
        fp16=True,
        logging_steps=50,
    )

    # Train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=noisy_domain,
        loss=train_loss,
    )

    trainer.train()

    print("Domain adaptation complete")
    return model

In [44]:
# Adapt model to banking domain
adapted_model = domain_adapt(general_model, domain_sentences, "domain_adapted")


Adapting to domain...

Adding noise to sentences...


Processing noisy data: 100%|██████████| 5000/5000 [00:00<00:00, 7093.08it/s]


Created 5000 noisy examples


Some weights of BertLMHeadModel were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.se

Step,Training Loss
50,8.532800
100,4.636600
150,3.835300
200,3.677100
250,3.469400
300,3.372200


Domain adaptation complete


**Stage 4: Re-evaluate on Domain Task**

In [45]:
# Evaluate adapted model
adapted_accuracy = evaluate_on_domain(adapted_model)


Evaluating on Banking77Classification...


Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Accuracy: 0.3908


Compare results.

In [46]:
print("\n" + "="*60)
print("Domain Adaptation Results:")
print(f"Baseline (general):  {baseline_accuracy:.4f}")
print(f"Adapted (banking):   {adapted_accuracy:.4f}")
print(f"Improvement:         {adapted_accuracy - baseline_accuracy:.4f}")


Domain Adaptation Results:
Baseline (general):  0.8004
Adapted (banking):   0.3908
Improvement:         -0.4096


### Questions

1. Did domain adaptation improve accuracy? Why would unsupervised adaptation help?

2. What makes a good domain for adaptation? When would adaptation not help?

3. Could you combine domain adaptation with supervised fine-tuning? How?

**About This Task:**

Real-world systems often combine multiple training strategies. This task chains TSDAE pretraining, supervised fine-tuning, and iterative evaluation.